# 과제 - 신경망을 이용한 손글씨 숫자 인식



## 1. 환경설정



In [ ]:
# Colab: ? ?? ?? ?? ????? (JWH ??? ?? ? ????? ??)
# GitHub URL? ???? ?? ??? ???? ?? ?? ???? ????.
import os
import sys
from pathlib import Path

GIT_URL = "https://github.com/Jungle-12-303/wk13_team2_mnist.git"
BRANCH = "JWH"
REPO_NAME = "wk13_team2_mnist"

if "google.colab" in sys.modules:
    repo_path = Path("/content") / REPO_NAME

    # ?? ??? ??? ??? ?? clone?? ?? JWH ??? ?? ??? ????.
    if repo_path.exists():
        os.chdir(repo_path)
        !git fetch origin
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
    else:
        os.chdir("/content")
        !git clone -b {BRANCH} {GIT_URL}
        os.chdir(repo_path)

    src_path = str(Path.cwd() / "src")
    if src_path in sys.path:
        sys.path.remove(src_path)
    sys.path.insert(0, src_path)
else:
    src_path = str(Path.cwd() / "src")
    if src_path in sys.path:
        sys.path.remove(src_path)
    sys.path.insert(0, src_path)

# ?? ???? main ??? ??? ???? ?? ? ?? ?? ??? ????.
for module_name in ["training", "network", "losses", "layers", "activations", "optimizers", "data"]:
    sys.modules.pop(module_name, None)

print("?? ?? ??:", Path.cwd())
print("?? ???:")
!git branch --show-current


## 2. 데이터 로드

In [ ]:
from data import load_mnist

(x_train, y_train), (x_test, y_test) = load_mnist()
print('Train:', x_train.shape, y_train.shape)
print('Test:', x_test.shape, y_test.shape)

## 3. 구현 및 테스트 통과 확인

`src/` 아래 역할별 파일의 **TODO**를 순서대로 구현한 뒤 아래 셀을 실행하세요.
- 주요 구현 파일: `activations.py`, `layers.py`, `losses.py`, `optimizers.py`, `network.py`, `training.py`
- 구현 파일은 역할별 모듈을 직접 import합니다. 예: `from network import NeuralNetwork`
- 개발 순서: 과제 안내문 참조
- 테스트: `tests/` 아래의 단계별 단위 테스트를 필요한 파일부터 실행합니다. 처음에는 전체 테스트보다 맡은 부분의 테스트 파일을 먼저 실행하세요.
    - ReLU만 확인: `TEST_TARGET = "tests/test_relu.py"`
    - 파일 안의 일부 테스트만 확인: `PYTEST_KEYWORD = "backward"`
    - 전체 테스트 확인: `TEST_TARGET = "tests/"`

In [ ]:
import subprocess
import sys
from pathlib import Path

# Colab/로컬 모두 현재 노트북 실행 위치를 저장소 루트로 사용합니다.
repo_dir = Path.cwd()

# 처음에는 자신이 구현 중인 부분의 테스트 파일만 실행하세요.
# 예: tests/test_relu.py, tests/test_affine.py, tests/test_training.py
TEST_TARGET = "tests/test_relu.py"

# 특정 이름이 들어간 테스트만 실행하고 싶을 때 사용합니다.
# 예: "backward". 전체 파일을 실행하려면 빈 문자열로 둡니다.
PYTEST_KEYWORD = ""

cmd = [sys.executable, "-m", "pytest", TEST_TARGET, "-v"]
if PYTEST_KEYWORD:
    cmd.extend(["-k", PYTEST_KEYWORD])

print("실행 경로:", repo_dir)
print("실행 명령:", " ".join(cmd))
result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    cwd=str(repo_dir)
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode == 0:
    print("\n선택한 테스트를 통과했습니다.")
else:
    print("\n선택한 테스트 중 실패가 있습니다.")


## 4. 모델·옵티마이저 생성 및 학습

In [ ]:
from network import NeuralNetwork
from optimizers import Adam
from training import train

model = NeuralNetwork(use_batchnorm=True, use_dropout=True)  # BatchNorm, Dropout 필수
optimizer = Adam(lr=0.001)

loss_history = train(model, optimizer, x_train, y_train, epochs=20, batch_size=128)

## 5. 평가 및 손실 커브

In [ ]:
from training import evaluate, plot_loss_history

acc, n_params = evaluate(model, x_test, y_test)
print(f'Test Accuracy: {acc:.2f}%')
print(f'Total Params: {n_params:,}')

plot_loss_history(loss_history)

## JWH Branch Setup

?? ??? ???? ?? ? ?? ?? ?????.
URL?? ???? ???? ??? `JWH` ??? ?? ??? ??? import ??? ????.


In [ ]:
import os
import sys
from pathlib import Path

REPO_NAME = "wk13_team2_mnist"
repo_dir = Path.cwd()
if not (repo_dir / "src").exists() and Path(f"/content/{REPO_NAME}/src").exists():
    repo_dir = Path(f"/content/{REPO_NAME}")
os.chdir(repo_dir)

!git fetch origin
!git checkout JWH
!git pull origin JWH

src_path = str(Path.cwd() / "src")
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)

for module_name in ["training", "network", "losses", "layers", "activations", "optimizers", "data"]:
    sys.modules.pop(module_name, None)

print("Ready on branch JWH")
!git branch --show-current


## Experiment Runner

?? ??? ?? MLP ?? ??? ?? ???? `results.csv`, `results.json`? ?????.
???? ??? ?? ??? ??? `configs[:1]`? baseline? ??? ?, ???? ?? configs? ?? ?????.


In [ ]:
from data import load_mnist
from training import DEFAULT_EXPERIMENT_CONFIGS, run_experiments, plot_compare_loss_histories

(x_train, y_train), (x_test, y_test) = load_mnist()

# ?? ??: baseline 1?? ??
configs = DEFAULT_EXPERIMENT_CONFIGS[:1]

# ?? ?? ?? ? ?? ?? ?????.
# configs = DEFAULT_EXPERIMENT_CONFIGS

results, histories = run_experiments(
    configs,
    x_train, y_train,
    x_test, y_test,
    csv_path="results.csv",
    json_path="results.json",
)

results
